# FlashEats — Class 6 Validation (Local Solved)

Solved locally against the submission repository. This notebook turns business assumptions into validation checks and a publishability decision.

In [1]:
from pathlib import Path
import json, sqlite3, sys, time, subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path(r"C:\Users\piyus\Downloads\FDE\flasheats-assignment-submission")]
    for p in candidates:
        if (p / "data" / "raw" / "sqlite" / "flasheats.db").exists():
            return p
    raise FileNotFoundError("Could not find the submission repo. Open this notebook from the submission folder or update ROOT.")

ROOT = find_root()
DB = ROOT / "data" / "raw" / "sqlite" / "flasheats.db"
DATA = ROOT / "data" / "raw" / "files"
OUT = ROOT / "outputs" / "classroom_challenges"
OUT.mkdir(parents=True, exist_ok=True)
con = sqlite3.connect(DB)
orders = pd.read_sql("SELECT * FROM orders", con)
orders["created_at"] = pd.to_datetime(orders["created_at"], errors="coerce")
orders["promised_eta"] = pd.to_datetime(orders["promised_eta"], errors="coerce")
orders["pickup_at"] = pd.to_datetime(orders["pickup_at"], errors="coerce")
orders["actual_delivery_at"] = pd.to_datetime(orders["actual_delivery_at"], errors="coerce")
orders = orders.drop_duplicates("order_id", keep="first")
outcomes = pd.read_csv(DATA / "order_outcomes.csv")
events = pd.read_csv(DATA / "order_events.csv")
tickets = pd.read_csv(DATA / "support_tickets.csv")
actions = pd.read_csv(DATA / "customer_app_actions.csv")
interventions = pd.read_csv(DATA / "order_interventions.csv")
restaurant_status = pd.read_csv(DATA / "restaurant_status.csv")
driver_events = json.loads((DATA / "driver_events.json").read_text(encoding="utf-8"))


## Challenge 1 — Validation contract for the 56% claim

In [2]:
delivered = orders[orders["final_status"].astype(str).str.lower().eq("delivered")]
contract = pd.DataFrame([
 {"assumption":"One row = one business order","test":"unique order_id after quarantine","status":"WARN" if orders["order_id"].duplicated().any() else "PASS","evidence":int(orders["order_id"].duplicated().sum())},
 {"assumption":"Delivered orders have completion time","test":"delivered actual_delivery_at non-null","status":"WARN" if delivered["actual_delivery_at"].isna().any() else "PASS","evidence":int(delivered["actual_delivery_at"].isna().sum())},
 {"assumption":"Promised ETA is valid","test":"promised_eta >= created_at","status":"WARN" if (orders["promised_eta"] < orders["created_at"]).any() else "PASS","evidence":int((orders["promised_eta"] < orders["created_at"]).sum())},
 {"assumption":"Event chronology is valid","test":"pickup_at <= actual_delivery_at","status":"WARN" if (orders["pickup_at"] > orders["actual_delivery_at"]).any() else "PASS","evidence":int((orders["pickup_at"] > orders["actual_delivery_at"]).sum())},
 {"assumption":"Late has an agreed definition","test":"compare any-delay and >10-minute definitions","status":"WARN","evidence":"definition ownership required"},
]); display(contract)

,assumption,test,status,evidence
0,One row = one business order,unique order_id after quarantine,PASS,0
1,Delivered orders have completion time,delivered actual_delivery_at non-null,WARN,37
2,Promised ETA is valid,promised_eta >= created_at,WARN,4
3,Event chronology is valid,pickup_at <= actual_delivery_at,WARN,5
4,Late has an agreed definition,compare any-delay and >10-minute definitions,WARN,definition ownership required


## Challenge 2 — Three late definitions

In [3]:
x = delivered.dropna(subset=["promised_eta","actual_delivery_at"]).copy(); x["delay_min"]=(x["actual_delivery_at"]-x["promised_eta"]).dt.total_seconds()/60
definitions = pd.DataFrame([
 {"definition":"delay > 0 minutes","late_rate_pct":round((x["delay_min"]>0).mean()*100,2),"business_meaning":"any miss of promised ETA"},
 {"definition":"delay > 10 minutes","late_rate_pct":round((x["delay_min"]>10).mean()*100,2),"business_meaning":"meaningful customer-impact miss"},
 {"definition":"historical delivered/non-null population","late_rate_pct":round((x["delay_min"]>0).mean()*100,2),"business_meaning":"only comparable measurable deliveries"},
]); display(definitions); print("Recommendation: publish the agreed definition in the metric contract; operations and the client KPI owner must approve it.")

,definition,late_rate_pct,business_meaning
0,delay > 0 minutes,57.03,any miss of promised ETA
1,delay > 10 minutes,24.27,meaningful customer-impact miss
2,historical delivered/non-null population,57.03,only comparable measurable deliveries


Recommendation: publish the agreed definition in the metric contract; operations and the client KPI owner must approve it.


## Challenge 3 — Category validation

In [4]:
for label, df, col in [("final_status",orders,"final_status"),("traffic_bucket",orders,"traffic_bucket"),("restaurant status",restaurant_status,"status"),("support category",tickets,"category")]:
    print("\n"+label); print(sorted(df[col].dropna().astype(str).unique().tolist()))
print("Safe normalization: case and whitespace can be standardized. Semantic mappings such as handoff versus handed_off require owner confirmation.")


final_status
['Delivered', 'cancelled', 'delivered']

traffic_bucket
['HIGH', 'high', 'low', 'medium', 'severe']

restaurant status
['READY', 'Ready', 'handed_off', 'handoff', 'preparing', 'ready', 'ready ', 'unknown']

support category
['ETA issue', 'Late Delivery', 'driver_not_moving', 'eta_changed', 'late_delivery', 'late_delivery ', 'ready_but_waiting', 'restaurant_delay', 'status_mismatch']
Safe normalization: case and whitespace can be standardized. Semantic mappings such as handoff versus handed_off require owner confirmation.


## Challenge 4 — Cross-source integrity

In [5]:
restaurants = pd.read_sql("SELECT * FROM restaurants", con); drivers = pd.read_sql("SELECT * FROM drivers", con)
checks = [
 ("orders.restaurant_id → restaurants", orders["restaurant_id"].isin(restaurants["restaurant_id"])),
 ("orders.driver_id → drivers", orders["driver_id"].isin(drivers["driver_id"])),
 ("tickets.order_id → orders", tickets["order_id"].isin(orders["order_id"])),
 ("restaurant_status.order_id → orders", restaurant_status["order_id"].isin(orders["order_id"])),]
mapping = pd.DataFrame([{"relationship":n,"coverage_pct":round(s.mean()*100,2),"status":"PASS" if s.all() else "WARN","unmapped":int((~s).sum())} for n,s in checks]); display(mapping)
print("A small unmapped percentage is only acceptable after checking whether those records affect the decision being made.")

,relationship,coverage_pct,status,unmapped
0,orders.restaurant_id → restaurants,99.81,WARN,3
1,orders.driver_id → drivers,99.81,WARN,3
2,tickets.order_id → orders,98.51,WARN,3
3,restaurant_status.order_id → orders,100.00,PASS,0


A small unmapped percentage is only acceptable after checking whether those records affect the decision being made.


## Challenge 5 — Freshness is an SLA question

In [6]:
rs = restaurant_status.copy(); rs["last_updated_at"]=pd.to_datetime(rs["last_updated_at"],errors="coerce")
joined=rs.merge(orders[["order_id","created_at","actual_delivery_at"]],on="order_id",how="left"); joined["update_lag_min"]=(joined["last_updated_at"]-pd.to_datetime(joined["created_at"],errors="coerce")).dt.total_seconds()/60
display(joined["update_lag_min"].describe())
print("Assessment: freshness may be adequate for weekly analytics, but live ETA requires an explicit SLA and current update timestamp at decision time; restaurant accountability requires agreeing which status event is authoritative.")

count    502.000000
mean      15.922311
std       41.129559
min     -354.000000
25%       12.000000
50%       20.000000
75%       28.000000
max       35.000000
Name: update_lag_min, dtype: float64

Assessment: freshness may be adequate for weekly analytics, but live ETA requires an explicit SLA and current update timestamp at decision time; restaurant accountability requires agreeing which status event is authoritative.


## Challenge 6 — Validation gate

In [7]:
validation_report = {"business_grain":"WARN","timestamp_chronology":"WARN","kpi_definition":"WARN","category_semantics":"PASS","cross_source_mapping":"PASS" if mapping["status"].eq("PASS").all() else "WARN","freshness":"UNKNOWN","publish_56_percent":"WARN"}
print(json.dumps(validation_report, indent=2)); print("Decision: do not publish an unqualified 56% claim. Publish only with the denominator, late definition, exclusions, and known data-quality caveats attached.")

{
  "business_grain": "WARN",
  "timestamp_chronology": "WARN",
  "kpi_definition": "WARN",
  "category_semantics": "PASS",
  "cross_source_mapping": "WARN",
  "freshness": "UNKNOWN",
  "publish_56_percent": "WARN"
}
Decision: do not publish an unqualified 56% claim. Publish only with the denominator, late definition, exclusions, and known data-quality caveats attached.


## Final takeaway

The validation gate is WARN: the data is useful for a transparent exploratory decision, but the headline KPI needs an agreed definition, duplicate/timestamp remediation, chronology review, and freshness ownership before it is treated as a production-grade number.